# Evaluate the book recommendation RAG pipeline

Run every question in the LangSmith `rag-evaluation-dataset` through the RAG pipeline, then score **context recall, context precision, faithfulness, and response relevancy**.

The notebook is ready to run; the full API evaluation has not been executed.

Use the **ai-engineer** kernel and start Docker Compose before running this notebook from top to bottom. This uses the project's installed Ragas **0.3.1** API and makes OpenAI calls for generation and evaluation.

| Metric | What it checks | Inputs used |
| --- | --- | --- |
| Context recall | How much of the reference answer is supported by retrieval? | Question, retrieved passages, reference answer |
| Context precision | Are passages useful for the reference answer ranked first? | Question, retrieved passages, reference answer |
| Faithfulness | Are the generated answer's claims supported by retrieval? | Question, generated answer, retrieved passages |
| Response relevancy | Does the generated answer address the question? | Question, generated answer |

These four metrics do not directly measure whether the generated answer matches the reference answer. In particular, faithfulness measures grounding rather than answer correctness.

## 1. Setup

Load the repository's `.env` before creating either API client. The answer model is the same one used by the app; the judge model scores its outputs.

Ragas 0.3.1 patches the event loop during import. Apply `nest_asyncio2` first so its Python 3.14-compatible implementation is used.

In [1]:
# Apply the Python 3.14-compatible patch before importing Ragas.
import nest_asyncio2
nest_asyncio2.apply()

import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from openai import OpenAI
from qdrant_client import QdrantClient
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ResponseRelevancy,
)
from ragas.run_config import RunConfig
from tqdm import tqdm

load_dotenv(override=True)

DATASET_NAME = "rag-evaluation-dataset"
ANSWER_MODEL = "gpt-5-nano"
JUDGE_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
TOP_K = 5

openai_client = OpenAI()
langsmith_client = Client()
qdrant_client = QdrantClient(url="http://localhost:6333")

pd.set_option("display.max_colwidth", 80)

/Users/benny/AI-Engineer/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load all reference examples

Each example stores the question in `inputs["question"]` and the reference answer in `outputs["ground_truth"]`. Fetch the full dataset once, with no ten-example limit.

In [18]:
dataset = langsmith_client.read_dataset(dataset_name=DATASET_NAME)
examples = list(langsmith_client.list_examples(dataset_id=dataset.id))

print(f"Loaded {len(examples)} examples from {dataset.name}")
display(pd.DataFrame([
    {"question": example.inputs["question"], "reference": example.outputs["ground_truth"]}
    for example in examples
]).head(3))

Loaded 37 examples from rag-evaluation-dataset


,question,reference
0,Can you tell me if any books are about a teacher or school setting turned in...,"'This Is Your Life' (ID 080214134X) features Jimmy Conway, a teacher who bec..."
1,Which titles would be relevant if I asked for books about maritime adventure...,'The Demon's Possession' (ID 098357331X) features a merchant ship and pirate...
2,Are there any books in stock that center on code breakers or spy romance set...,'A Christmas Code' (ID 0989939197) is a Regency-era spy romance involving Co...


## 3. RAG pipeline

The retrieval, prompt, and answer generation follow the app's pipeline. LangSmith tracing code is omitted here so the functions only perform the RAG steps. The OpenAI and Qdrant clients created above are reused for every question.

In [19]:
def get_embedding(text, model=EMBEDDING_MODEL):
    response = openai_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding


def retrieve_similar_books(query, qdrant_client, top_k=5):

    query_embedding = get_embedding(query)
    
    search_result = qdrant_client.query_points(
        collection_name="amazon_books_collection_sampled",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context.append(result.payload['description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
    
    return {
        "context_ids": retrieved_context_ids,
        "context": retrieved_context,
        "similarity_scores": similarity_scores,
        "ratings": retrieved_context_ratings
    }

In [20]:
def process_context(retrieved_context):
    formatted_context = ""

    for id, chunk, rating in zip(retrieved_context["context_ids"], retrieved_context["context"], retrieved_context["ratings"]):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n\n"
    return formatted_context


def build_prompt(preprocessed_context, user_query):
    prompt = f"""
    You are a book recommendation assistant that can answer questions about the books in stock.

    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products
    
    Context:
    {preprocessed_context}
    
    Question: {user_query}
    
    Answer:
    """
    return prompt


def generate_answer(prompt):
    response = openai_client.chat.completions.create(
        model = ANSWER_MODEL,
        messages =[{"role": "system", "content": prompt}],
        reasoning_effort = 'minimal'
    )

    return response.choices[0].message.content

In [21]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_similar_books(question, qdrant_client, top_k)
    processed_context = process_context(retrieved_context)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        'answer': answer,
        'question': question,
        'retrieved_context': retrieved_context,
        'retrieved_context_ids': retrieved_context["context_ids"],
        'similarity_scores': retrieved_context["similarity_scores"]
    }
    return final_result

## 4. Generate the evaluation inputs

For each question, keep the pipeline's answer, its actual retrieved passages, and the separate reference answer. Include the IDs and ratings in each passage, matching the evidence supplied to the answer model. Retrieval order is preserved for context precision.

The rows use Ragas's field names: `user_input`, `response`, `retrieved_contexts`, and `reference`. Save them to `rag_eval_inputs.json` so scoring can be rerun without regenerating answers.

In [22]:
evaluation_rows = []

for example in tqdm(examples, desc="Running RAG"):
    prediction = rag_pipeline(example.inputs["question"], top_k=TOP_K)
    retrieved = prediction["retrieved_context"]
    retrieved_contexts = [
        f"ID: {book_id}, Rating: {rating}, Description: {description}"
        for book_id, rating, description in zip(
            retrieved["context_ids"], retrieved["ratings"], retrieved["context"]
        )
    ]
    evaluation_rows.append({
        "user_input": example.inputs["question"],
        "response": prediction["answer"],
        "retrieved_contexts": retrieved_contexts,
        "reference": example.outputs["ground_truth"],
    })

Path("rag_eval_inputs.json").write_text(
    json.dumps(evaluation_rows, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Saved {len(evaluation_rows)} generated answers and their retrieved passages")
display(pd.DataFrame(evaluation_rows)[["user_input", "response"]].head(3))

Running RAG: 100%|██████████| 37/37 [01:23<00:00,  2.27s/it]

Saved 37 generated answers and their retrieved passages


,user_input,response
0,Can you tell me if any books are about a teacher or school setting turned in...,Yes. The Job of the Wasp: A Novel is set in an isolated school for orphaned ...
1,Which titles would be relevant if I asked for books about maritime adventure...,"Here are titles from the available products that fit maritime adventures, pi..."
2,Are there any books in stock that center on code breakers or spy romance set...,Yes. The available books that fit a code-breaker or spy romance theme set ag...


## 5. Configure the Ragas judge

`LangchainLLMWrapper` adapts the judge model to this installed Ragas version. Response relevancy also needs embeddings to compare the original question with questions generated from the answer. The embedding model name is `text-embedding-3-small`.

The metric classes below use Ragas's current single-turn sample fields, rather than the older tutorial's `question` / `answer` / `contexts` names.

In [2]:
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model=JUDGE_MODEL, temperature=0)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model=EMBEDDING_MODEL)
)

metrics = [
    LLMContextRecall(),
    LLMContextPrecisionWithReference(name="context_precision"),
    Faithfulness(),
    ResponseRelevancy(name="response_relevancy"),
]

## 6. Score all examples

This cell reads the saved generated answers. To try a different judge, rerun the setup, judge configuration, and scoring cells; there is no need to rerun answer generation.

**If you saw `Timeout should be used inside a task`, restart the ai-engineer kernel once.** The old `nest_asyncio` patch cannot be replaced safely in an already-patched kernel. Then run sections **1, 5, 6, and 7** to reuse `rag_eval_inputs.json`; skip answer generation in section 4.

API errors are raised so they can be fixed. A metric can still return `NaN` when it cannot score an example, such as faithfulness for an answer with no factual claims; that is not a zero score.

In [3]:
evaluation_rows = json.loads(Path("rag_eval_inputs.json").read_text(encoding="utf-8"))
evaluation_dataset = EvaluationDataset.from_list(evaluation_rows)

results = evaluate(
    dataset=evaluation_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=RunConfig(max_workers=4, max_retries=2),
    raise_exceptions=True,
)

Evaluating: 100%|██████████| 148/148 [02:30<00:00,  1.02s/it]


## 7. Inspect the scores

The summary gives the mean and number of scored examples for each metric. Means exclude missing values; the counts make that visible. Inspect individual rows to see which questions need attention rather than treating the four means as a single accuracy score.

In [4]:
results_df = results.to_pandas()
metric_columns = [metric.name for metric in metrics]

summary = results_df[metric_columns].agg(["mean", "count"]).T
summary = summary.rename(columns={"count": "scored_examples"})
summary["scored_examples"] = summary["scored_examples"].astype(int)
display(summary.round(3))

results_df.to_csv("rag_eval_scores.csv", index=False)
print(f"Saved scores for {len(results_df)} examples to rag_eval_scores.csv")

,mean,scored_examples
context_recall,0.786,37
context_precision,0.746,37
faithfulness,0.858,37
response_relevancy,0.620,37


Saved scores for 37 examples to rag_eval_scores.csv


In [5]:
display(results_df[["user_input", *metric_columns]].round(3))

,user_input,context_recall,context_precision,faithfulness,response_relevancy
0,Can you tell me if any books are about a teacher or school setting turned in...,0.000,0.000,1.000,0.312
1,Which titles would be relevant if I asked for books about maritime adventure...,0.333,0.250,0.900,0.916
2,Are there any books in stock that center on code breakers or spy romance set...,1.000,1.000,0.923,0.925
3,Do you have novels that pair romance with mountain or wilderness settings?,0.000,1.000,0.778,0.869
4,Which of your books are explicitly unanswerable from the chunks provided — f...,1.000,1.000,0.429,0.000
5,Are there any books that combine magic artifacts with action-adventure and a...,1.000,0.250,1.000,0.650
6,Can I find any novels here that explore immigrant experiences in America thr...,0.500,1.000,0.643,0.813
7,Is there any book on the list that deals with a protagonist who is a concert...,1.000,1.000,1.000,0.297
8,Which books on your list are explicitly unanswerable from the chunks — for e...,1.000,1.000,1.000,0.000
9,Do you have any crime or detective novels set in South Korea during the 1970s?,1.000,1.000,1.000,0.613


### Look at one answer in detail

Change `row_index` to inspect a question, the generated answer, the reference, and the exact retrieved evidence behind its scores.

In [6]:
row_index = 0
row = results_df.iloc[row_index]

print("QUESTION:\n", row["user_input"])
print("\nGENERATED ANSWER:\n", row["response"])
print("\nREFERENCE ANSWER:\n", row["reference"])
print("\nRETRIEVED PASSAGES:\n", "\n\n".join(row["retrieved_contexts"]))
display(row[metric_columns].astype(float).round(3))

QUESTION:
 Can you tell me if any books are about a teacher or school setting turned into chaos or surreal events?

GENERATED ANSWER:
 Yes. The Job of the Wasp: A Novel is set in an isolated school for orphaned boys, where a new arrival uncovers secrets and the atmosphere becomes chaotic and eerie as corpses turn up. It fits a school setting turned surreal/haunted by darker forces.

REFERENCE ANSWER:
 'This Is Your Life' (ID 080214134X) features Jimmy Conway, a teacher who becomes a stand-up sensation through a comedic deception. 'Bittersweet' (ID 034544597X) includes family and community dynamics but is less focused on a school meltdown; for surreal school events, 'Music for Torching' (ID 068817762X) features a hostage situation at a boys' school (not in chunk list above).

RETRIEVED PASSAGES:
 ID: 149450863X, Rating: 3.7, Description: Dark Screams: Volume One (Dark Screams, 1)"Weeds" by Stephen King: When a meteorite lands on his property, Jordy Verrill envisions an easy payday. Unfo

context_recall        0.000
context_precision     0.000
faithfulness          1.000
response_relevancy    0.312
Name: 0, dtype: float64

### Reading the results

- Low **context recall**: the retrieved passages do not support much of the reference answer.
- Low **context precision**: passages judged useful for the reference answer are missing or ranked behind less useful ones.
- Low **faithfulness**: the generated answer contains claims unsupported by the retrieved passages.
- Low **response relevancy**: the answer does not address the question well; refusals can also score low even when refusing is appropriate.

These are model-judged measurements, so scores can vary between runs. This dataset's references were generated synthetically; review a low-scoring example's reference before assuming the pipeline is wrong.

For a retrieval or prompt experiment, change the pipeline or `TOP_K`, regenerate the evaluation inputs, then rerun scoring. For a judge experiment, keep the saved inputs and only rerun the judge and scoring cells.

Ragas 0.3.1 documentation: [evaluate](https://docs.ragas.io/en/v0.3.1/references/evaluate/), [context recall](https://docs.ragas.io/en/v0.3.1/concepts/metrics/available_metrics/context_recall/), [context precision](https://docs.ragas.io/en/v0.3.1/concepts/metrics/available_metrics/context_precision/), [faithfulness](https://docs.ragas.io/en/v0.3.1/concepts/metrics/available_metrics/faithfulness/), [response relevancy](https://docs.ragas.io/en/v0.3.1/concepts/metrics/available_metrics/answer_relevance/).